## ReasoningChain batch runner (Kaggle-friendly)

This notebook runs `ReasoningChain` over the full `train.csv` (~9000 rows) and writes results as **JSONL** (one `DataPoint` per line).

- **Input**: Kaggle-attached dataset under `/kaggle/input/...` (auto-detected) or fallback to `kagglehub` download.
- **Output**: `/kaggle/working/reasoning_chain_results.jsonl`
- **Resume**: if the JSONL exists, already-seen `id`s are skipped.

Choose backend:
- **HF (Transformers)**: recommended for Kaggle.
- **Ollama**: only if you have an Ollama server reachable (typically local dev, not Kaggle).


In [ ]:
!pip install --upgrade git+https://github.com/huggingface/transformers.git


In [ ]:
import sys
import os

# Replace 'your-dataset-name' with the actual name of the dataset you uploaded
# You can find the exact path by clicking the 'Data' tab on the right sidebar
repo_path = '/kaggle/input/datasets/kyinxu/nemotron-reasoning-chain/src'

if os.path.exists(repo_path):
    sys.path.append(repo_path)
    print("Successfully added src to path!")
else:
    print("Path not found. Check the sidebar for the correct dataset folder name.")

In [ ]:
import json
import logging
from pathlib import Path
from typing import Optional, Set

import polars as pl

from chain import ReasoningChain, convert_to_entry
from registry.datasets import get_train_csv_path
from registry.models import get_model, get_transformers_chat_model

logging.basicConfig(level=logging.INFO)


In [ ]:
# --- Config ---

# Backend: set to "hf" for Kaggle, "ollama" for local dev
BACKEND = "hf"  # "hf" | "ollama"

# HF model id/path (required when BACKEND == "hf")
HF_MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"

# Ollama registry model id (required when BACKEND == "ollama")
OLLAMA_MODEL_ID = "nemotron3-4b"

# Generation params for HF
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.2
TOP_P = 0.95
TRUST_REMOTE_CODE = True

# Optional: limit rows for a quick test
LIMIT: Optional[int] = None  # e.g. 10

# Output location
OUT_PATH = (
    Path("/kaggle/working/reasoning_chain_results.jsonl")
    if Path("/kaggle/working").is_dir()
    else Path("reasoning_chain_results.jsonl")
)

# Resume behavior
RESUME = True


In [ ]:
def load_done_ids(path: Path) -> Set[str]:
    done: Set[str] = set()
    if not path.is_file():
        return done
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                done.add(str(json.loads(line)["id"]))
            except Exception:
                continue
    return done


def make_model():
    if BACKEND == "ollama":
        return get_model(OLLAMA_MODEL_ID)
    if BACKEND == "hf":
        if not HF_MODEL:
            raise ValueError("HF_MODEL must be set when BACKEND='hf'")
        return get_transformers_chat_model(
            HF_MODEL,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            trust_remote_code=TRUST_REMOTE_CODE,
        )
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}")


In [ ]:
# Load dataset
train_csv = get_train_csv_path()
print("train_csv:", train_csv)

df = pl.read_csv(train_csv)
if LIMIT is not None:
    df = df.head(LIMIT)

print("rows:", df.height)


In [ ]:
# Run and stream JSONL
done = load_done_ids(OUT_PATH) if RESUME else set()
print("resume:", RESUME, "already_done:", len(done), "output:", OUT_PATH)

model = make_model()
chain = ReasoningChain(model, verbose=True)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
written = 0
with OUT_PATH.open("a", encoding="utf-8") as fh:
    for row in df.iter_rows(named=True):
        eid = str(row["id"]).strip()
        if eid in done:
            continue
        entry = convert_to_entry(pl.DataFrame([row]))
        dp = chain.run(entry)
        fh.write(json.dumps(dp.model_dump(), ensure_ascii=False) + "\n")
        fh.flush()
        done.add(eid)
        written += 1
        if written % 10 == 0:
            print("written:", written, "last_id:", eid)

print("done. written:", written)
